# 01. Count components

**Tier:** Deep. **Why you'd use this:** You have `n` nodes and an undirected edge list. You need how many connected pieces remain after every edge merges its ends.

**After this lesson you can:** find roots with path compression; union components and count how many roots remain.
About 25 minutes. **Uses:** [Loops](../../course/03-control-flow/02-loops-lesson.ipynb), [Lists](../../course/04-collections/01-lists-lesson.ipynb)


## Quick review

1. What does `find` return for a node?
2. When does a union decrease the component count?

<details><summary>Answers</summary>

The root label of its component. Only when the two ends currently have different roots.
</details>


## Recognize it

The input is nodes plus undirected edges, and the question is about connectivity: component count, same-component checks, or redundant edges that close a cycle.

Signals: "connected components", "unite", "do these nodes share a group". A look-alike that is not this pattern: shortest path or leveled distance — that needs BFS or Dijkstra, not only merges.

If the graph is directed, plain undirected union-find is the wrong model unless you first forget directions on purpose. Strong connectivity needs a different algorithm.


## Why it works this way

Union-find keeps a parent pointer for every node. At the start each node is its own parent, so there are `n` components. `find(x)` walks parent links to the root. Path compression sets each visited node straight to that root on the way back, so later finds are short. `union(a, b)` finds both roots and links one to the other when they differ. Each successful link merges two components, so the running count drops by one.

The structure answers "same component?" without rebuilding a graph walk for every query. After path compression (and optionally union by rank), the amortized cost per operation is tiny — inverse Ackermann, which behaves like a constant on challenge-sized graphs. Memory is one parent array, and optionally a rank array, beyond the edge list.

A depth-first or breadth-first walk on an adjacency list also counts components in linear time when you only ask once after all edges are known. Union-find shines when merges and queries interleave, or when you want to detect the first edge that joins an already-connected pair. See [Lists](../../internals/03-lists/01-dynamic-array-lesson.ipynb) for the parent array as a flat mutable table of integers.

Forgetting to `find` before linking leaves shallow bugs: you may hang a whole tree under a non-root and later disagree about membership. Forgetting to decrease the count on a real merge reports too many components. Both mistakes are silent.

The design reason for parents instead of an adjacency list is the operation mix. You rarely need neighbors; you need "are these already united?" and "unite them if not." A forest of parents answers that with almost-constant updates, while rebuilding components with DFS after every edge would rescan the growing graph. When the only question is a final component count on a static edge list, DFS and union-find both finish in linear time — pick the one that matches how queries arrive.


## When to use it

Use union-find when connectivity merges stream in and you need component counts or same-set tests along the way. Prefer a single DFS or BFS when the graph is fixed and you only need components once. Prefer BFS for shortest paths. Prefer a full adjacency DFS when you need the actual path or the set of nodes in each piece for later processing.

Prefer Kruskal-style sorting of edges by weight plus union-find when building a minimum spanning tree. Prefer BFS layers when distance, not only membership, is the answer.


## Brute force, and why it's slow

Rebuild connectivity with DFS from every node for each edge batch and count visited groups. Here: DFS count after all edges, measuring node touches.


In [1]:
def count_dfs(n, edges):
    graph = {i: [] for i in range(n)}
    for a, b in edges:
        graph[a].append(b)
        graph[b].append(a)
    seen = set()
    touches = 0

    def walk(node):
        nonlocal touches
        seen.add(node)
        touches += 1
        for nxt in graph[node]:
            if nxt not in seen:
                walk(nxt)

    parts = 0
    for node in range(n):
        if node not in seen:
            parts += 1
            walk(node)
    return parts, touches


print(count_dfs(5, [[0, 1], [1, 2], [3, 4]]))

(2, 5)


## Key insight

Each node starts as its own component. Every edge whose ends have different roots merges them and decreases the count by one. Same-root edges are redundant for connectivity and leave the count unchanged.


## Worked example

Nodes are labeled `0 .. n - 1`. Edges are undirected pairs. Return how many connected components exist after all unions.


In [2]:
def count_components(n: int, edges: list[list[int]]) -> int:
    # 1. Each node begins as its own root.
    parent = list(range(n))
    parts = n

    def find(x: int) -> int:
        # 2. Climb to the root; compress the path.
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for a, b in edges:
        ra, rb = find(a), find(b)
        # 3. Link different roots and drop one component.
        if ra != rb:
            parent[rb] = ra
            parts -= 1
    return parts


print(count_components(5, [[0, 1], [1, 2], [3, 4]]))
print(count_components(3, [[0, 1], [1, 2], [0, 2]]))

2
1


## Trace

`n = 5`, edges `[0-1], [1-2], [3-4]`:

| edge | roots before | merge? | parts |
|---|---|---|---|
| start | — | — | 5 |
| 0-1 | 0, 1 | yes | 4 |
| 1-2 | 0, 2 | yes | 3 |
| 3-4 | 3, 4 | yes | **2** |

Components: `{0,1,2}`, `{3,4}`.


## Cost

Almost-constant work per edge after compressions, so time grows with `n + len(edges)` in practice. Memory is the parent list of length `n`. DFS also linear once, without supporting cheap interleaved queries. Rank or size heuristics keep trees shallow before compression finishes the job.


## Variants

Not solved here: [Redundant Connection](../../challenges/problems/50-redundant-connection.ipynb), [Number of Islands](../../challenges/problems/12-number-of-islands.ipynb), [Course Schedule](../../challenges/problems/33-course-schedule.ipynb).


## Common mistake

This bug is silent. The code links `parent[b] = a` without finding roots first. Later finds disagree, and the count never drops correctly on some graphs. On a chain it may still look plausible; on `[0,1], [1,2], [0,2]` with a broken linker the printed parts stay too high. Demo: skip the root check and never decrement.


In [3]:
def count_components_wrong(n, edges):
    parent = list(range(n))
    parts = n
    for a, b in edges:
        parent[b] = a  # no find; may hang under a non-root
        # forgot parts -= 1 on a real merge
    return parts


print(count_components_wrong(3, [[0, 1], [1, 2], [0, 2]]))

3


The fix finds both roots, links only when they differ, and decrements `parts`. The buggy cell prints `3` for `n = 3` with those edges because it never merges the linked roots.


In [4]:
print(count_components(3, [[0, 1], [1, 2], [0, 2]]))

1


## Python idioms

Half-path compression `parent[x] = parent[parent[x]]` is a short loop body. Union by rank adds a `rank` list and links the lower under the higher to keep trees shallow.


In [5]:
def count_components_ranked(n: int, edges: list[list[int]]) -> int:
    parent = list(range(n))
    rank = [0] * n
    parts = n

    def find(x: int) -> int:
        if parent[x] != x:
            parent[x] = find(parent[x])
        return parent[x]

    for a, b in edges:
        ra, rb = find(a), find(b)
        if ra == rb:
            continue
        if rank[ra] < rank[rb]:
            parent[ra] = rb
        elif rank[ra] > rank[rb]:
            parent[rb] = ra
        else:
            parent[rb] = ra
            rank[ra] += 1
        parts -= 1
    return parts


print(count_components_ranked(5, [[0, 1], [1, 2], [3, 4]]))

2


## Recap

- Parents start as identity; each merge drops one component.
- Always `find` before you link.
- Path compression keeps later finds short.

Next: [practice](01-count-components-practice.ipynb).
